# VLA 연구 · 13회차 — 숫자 조건 VLM → VLA 엔드투엔드 파이프라인

| 항목 | 내용 |
|---|---|
| 오늘 목표 | 숫자가 찍힌 **동일한 정육면체**를 VLM이 읽고 VLA가 옮기는 파이프라인을 끝까지 돌린다 |
| 핵심 질문 | 색·모양 단서를 없앴을 때 **1단계(VLM)와 2단계(VLA)의 성능이 각각 얼마나 떨어지는가?** |
| 환경 | MuJoCo + Franka Panda, 관측·액션은 **LIBERO 호환** 규격 (12회차와 동일) |
| 산출물 | 색 vs 숫자 비교표, 오류 분석(지각/추론 분리), 숫자 조건 rollout GIF, 데이터셋 준비 |

### 12회차와 달라진 점

12회차에서는 **색과 모양**이 서로 다른 10개 물체를 다뤘습니다.
VLM은 "빨간 블록"처럼 색만 보면 물체를 구분할 수 있었습니다.

이번에는 물체 10개가 **전부 똑같은 흰 정육면체**이고, 면에 찍힌 **숫자(1~10)**만이
유일한 구분 단서입니다. 색도 크기도 모양도 같으므로:

- VLM이 숫자를 **읽지 못하면** 계획을 세울 수 없고
- VLA가 지시문의 숫자를 **알아보지 못하면** 어떤 큐브를 집어야 하는지 모릅니다

이 조건에서 성능이 떨어진다면 그건 "옮기는 게 어려워서"가 아니라
**"무엇을 옮길지 알아보지 못해서"** 입니다.

---
## SECTION 0 | 설치와 코드 연결

12회차와 동일합니다. `vla_pipeline` 폴더가 팀 공유 드라이브에 있어야 합니다.

```
MyDrive/VLA_Team_2026/
└── vla_pipeline/          <- make_student_zip.py 로 만든 zip 을 푼 것
    ├── vlapipe/  scripts/  assets/  notebooks/
    └── README.md ...
```

In [ ]:
# 0-1. 패키지 설치 (Colab 최초 1회, 이후 런타임 재시작)
!pip install -q 'mujoco>=3.2,<4.0' h5py imageio imageio-ffmpeg 'transformers>=4.50' accelerate num2words torchvision Pillow
print('설치 완료 — Runtime > Restart session 후 0-2 부터 실행하세요.')

In [ ]:
# 0-2. Drive 마운트 + 코드를 런타임 로컬로 복사
import os, sys, shutil
os.environ['MUJOCO_GL'] = 'egl'
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    TEAM = Path('/content/drive/MyDrive/VLA_Team_2026')
    SRC = TEAM / 'vla_pipeline'
    assert SRC.exists(), f'공유 드라이브에 없습니다: {SRC}'
    PKG = Path('/content/vla_pipeline')
    # Preserve existing runtime data. Use a fresh runtime to change input datasets.
    shutil.copytree(SRC, PKG, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('.venv', 'outputs', '__pycache__', '.git'))
    print('코드 사본 :', PKG)
    print('원본      :', SRC, '(읽기만 함)')
else:
    PKG = Path.cwd().parent
    TEAM = PKG.parent

assert PKG.exists(), f'경로를 찾을 수 없습니다: {PKG}'
sys.path.insert(0, str(PKG))
os.chdir(PKG)
(PKG / 'outputs').mkdir(exist_ok=True)

n_ep = len(list((PKG / 'data' / 'episodes').glob('*.hdf5'))) if (PKG / 'data' / 'episodes').exists() else 0
print('작업 폴더 :', PKG)
print('녹화본    :', n_ep, '개')

In [ ]:
# 0-3. Panda 에셋 준비 (최초 1회, 약 40MB)
!python setup_assets.py

In [ ]:
# 0-4. 환경 점검
!python scripts/check_env.py

---
## SECTION 1 | 숫자 조건 씬 살펴보기

색 조건과 숫자 조건의 차이를 눈으로 확인합니다.

| | 색 조건 (`color`) | 숫자 조건 (`number`) |
|---|---|---|
| 물체 | 색·크기·모양이 다른 10개 | **전부 똑같은 흰 정육면체** 10개 |
| 구분 단서 | 색 + 모양 | 면에 찍힌 **숫자 1~10** 뿐 |
| 정돈 규칙 | 긴 블록→오른쪽 / 정육면체·타일→왼쪽 | **1~5→오른쪽 / 6~10→왼쪽** |
| 지시문 | "Pick up the red block …" | "Pick up the cube number 7 …" |

**바뀌지 않는 것**: 테이블·구역·조명·카메라·로봇, 배치, LIBERO 규격.

In [ ]:
# 1-1. 색 조건과 숫자 조건을 나란히 렌더
import numpy as np, matplotlib.pyplot as plt, pandas as pd
from vlapipe.env import SortingEnv
from vlapipe import config as C

# --- 색 조건 ---
env_color = SortingEnv(layout='messy', seed=3, image_size=C.IMAGE_SIZE, cameras=C.ALL_CAMERAS)
env_color.reset(layout='messy', seed=3)
img_color_256 = env_color.render('agentview', 256, 256)
img_color_512 = env_color.render('agentview', 512, 512)

# --- 숫자 조건 ---
from vlapipe import numbered
numbered.activate()
env_num = SortingEnv(layout='messy', seed=3, image_size=C.IMAGE_SIZE, cameras=C.ALL_CAMERAS)
env_num.reset(layout='messy', seed=3)
img_num_256 = env_num.render('agentview', 256, 256)
img_num_512 = env_num.render('agentview', 512, 512)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes[0, 0].imshow(img_color_256); axes[0, 0].set_title('Color  256px', fontsize=12); axes[0, 0].axis('off')
axes[0, 1].imshow(img_color_512); axes[0, 1].set_title('Color  512px', fontsize=12); axes[0, 1].axis('off')
axes[1, 0].imshow(img_num_256);   axes[1, 0].set_title(f'Number 256px  (face ~{numbered.face_pixels(256):.1f} px)', fontsize=12); axes[1, 0].axis('off')
axes[1, 1].imshow(img_num_512);   axes[1, 1].set_title(f'Number 512px  (face ~{numbered.face_pixels(512):.1f} px)', fontsize=12); axes[1, 1].axis('off')
plt.suptitle('Color condition (top) vs Number condition (bottom) — same seed, same layout', fontsize=13)
plt.tight_layout(); plt.show()

print('물체 세트:', C.ACTIVE_OBJECT_SET)
print('물체 이름:', C.OBJECT_NAMES)
print('정돈 규칙: 1~%d -> 오른쪽, %d~%d -> 왼쪽' % (C.NUMBER_SPLIT, C.NUMBER_SPLIT + 1, C.NUMBER_COUNT))
print('잘못 놓인:', env_num.misplaced_objects())
print()
print('위 그림에서 256px 숫자 조건(왼쪽 아래)의 숫자를 읽어 보세요.')
print('읽히지 않는다면, VLM 에게 256px 이미지를 주는 것은 숫자를 지우고 구분해 보라는 것과 같습니다.')

In [ ]:
# 1-2. 숫자 조건 — 카메라 4대로 보기
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, cam in zip(axes, C.ALL_CAMERAS):
    ax.imshow(env_num.render(cam, 400, 400))
    ax.set_title(cam, fontsize=10); ax.axis('off')
plt.suptitle('Number condition — all cameras (seed=3, messy)', fontsize=12)
plt.tight_layout(); plt.show()

print('agentview(top-down)에서 숫자가 가장 잘 보입니다.')
print('robot0_eye_in_hand(손목)에서도 가까이 가면 보이지만, 멀리서는 구분이 안 됩니다.')

---
## SECTION 2 | 1단계 — VLM 플래너로 숫자 읽기

숫자 조건의 핵심은 **VLM이 이미지에서 숫자를 읽을 수 있는가** 입니다.

두 가지 프롬프트를 비교합니다:

| 프롬프트 | VLM이 하는 일 | 규칙(1~5/6~10)은 누가 적용? |
|---|---|---|
| `number` | 숫자를 읽고 **목적지까지** 말한다 | VLM |
| `number_read` | 가운데 있는 숫자만 읽는다 | **코드** |

**읽는 법:**
- `number_read`도 낮다 → **숫자를 못 읽는 것** (지각 문제)
- `number_read`는 높은데 `number`가 낮다 → **읽기는 되는데 규칙을 못 쓰는 것** (추론 문제)
- 둘 다 색 조건보다 많이 낮다 → 색 조건의 점수에 **단서가 쉬워서 얻은 점수**가 섞여 있었다

In [ ]:
# 2-1. 정답 계획(규칙 플래너) 확인
from vlapipe.planner import RulePlanner, VLMPlanner, plan_accuracy, parse_plan, PROMPTS

env_num.reset(layout='messy', seed=3)
gold = RulePlanner(env_num).plan('테이블을 정리하시오')

print('=== 정답 계획 (규칙 플래너) ===')
for s in gold:
    print(f"  {s['step']}. {s['instruction']}")
    print(f"     {s['instruction_ko']}   -> ({s['object']}, {s['target_zone']})")
print(f'\n잘못 놓인 물체 {len(gold)}개')

In [ ]:
# 2-2. VLM(SmolVLM2)으로 숫자를 읽고 계획을 세우게 한다
VLM_PX = 512   # 숫자 조건은 512px 로 렌더해야 숫자가 보인다 (1-1 에서 확인)

vlm = VLMPlanner('HuggingFaceTB/SmolVLM2-500M-Video-Instruct', prompt_style='number')

env_num.reset(layout='messy', seed=3)
img = env_num.render('agentview', VLM_PX, VLM_PX)
raw = vlm.raw_response(img, '테이블을 정리하시오')

print('=== VLM 원문 (prompt_style="number") ===')
print(raw)

pred = parse_plan(raw, allow_default_zone=(vlm.prompt_style == 'number_read'))
print('\n=== 파싱된 계획 ===')
for s in pred:
    print(f"  {s['step']}. {s['object']} -> {s['target_zone']}")

print('\n=== 채점 ===')
acc = plan_accuracy(pred, gold)
for k, v in acc.items():
    print(f'  {k:14s} {v}')

In [ ]:
# 2-3. number_read 프롬프트 — 숫자만 읽게 하고 목적지는 코드가 결정
vlm.prompt_style = 'number_read'
env_num.reset(layout='messy', seed=3)
raw_read = vlm.raw_response(env_num.render('agentview', VLM_PX, VLM_PX), '테이블을 정리하시오')

print('=== VLM 원문 (prompt_style="number_read") ===')
print(raw_read)

pred_read = parse_plan(raw_read, allow_default_zone=True)
acc_read = plan_accuracy(pred_read, gold)
print('\n=== 채점 ===')
for k, v in acc_read.items():
    print(f'  {k:14s} {v}')

In [ ]:
# 2-4. 채점 루프 — 여러 시드 × 프롬프트 두 가지
SEEDS = [3, 7, 11, 19, 23]        # 수업 중에는 5개 (T4 에서 몇 분)
# SEEDS = list(range(30))         # 보고서용: 조건당 30씬

rows_num = []
for style in ['number', 'number_read']:
    vlm.prompt_style = style
    for seed in SEEDS:
        env_num.reset(layout='messy', seed=seed)
        g = RulePlanner(env_num).plan()
        p = parse_plan(vlm.raw_response(
            env_num.render('agentview', VLM_PX, VLM_PX), '테이블을 정리하시오'), allow_default_zone=(vlm.prompt_style == 'number_read'))
        a = plan_accuracy(p, g)
        rows_num.append({'프롬프트': style, 'seed': seed,
                         '정답수': a['n_gold'], '예측수': a['n_pred'], '맞춘수': a['correct'],
                         'recall': round(a['recall'], 2), 'precision': round(a['precision'], 2),
                         '빠뜨림': len(a['missed']), '없는물체': len(a['hallucinated']),
                         '구역틀림': len(a['wrong_zone'])})

df_num = pd.DataFrame(rows_num)
display(df_num)

In [ ]:
# 2-5. 색 조건도 같은 해상도(512px)로 다시 재기 — 공정한 비교
numbered.deactivate()
vlm.prompt_style = 'json'

rows_col = []
for seed in SEEDS:
    env_color.reset(layout='messy', seed=seed)
    g = RulePlanner(env_color).plan()
    p = parse_plan(vlm.raw_response(
        env_color.render('agentview', VLM_PX, VLM_PX), '테이블을 정리하시오'), allow_default_zone=(vlm.prompt_style == 'number_read'))
    a = plan_accuracy(p, g)
    rows_col.append({'프롬프트': 'json (color)', 'seed': seed,
                     '정답수': a['n_gold'], '예측수': a['n_pred'], '맞춘수': a['correct'],
                     'recall': round(a['recall'], 2), 'precision': round(a['precision'], 2),
                     '빠뜨림': len(a['missed']), '없는물체': len(a['hallucinated']),
                     '구역틀림': len(a['wrong_zone'])})

df_col = pd.DataFrame(rows_col)

# 숫자 조건으로 다시 돌아간다 (이후 SECTION 에서 숫자 조건을 쓰므로)
numbered.activate()
env_num = SortingEnv(layout='messy', seed=3, image_size=C.IMAGE_SIZE, cameras=C.ALL_CAMERAS)

In [ ]:
# 2-6. 세 조건 비교 요약표 (micro 평균)
def micro(df, label):
    c, g, n = int(df['맞춘수'].sum()), int(df['정답수'].sum()), int(df['예측수'].sum())
    return {'조건': label, '씬수': len(df), '정답수': g, '예측수': n, '맞춘수': c,
            'recall': round(c / g, 2) if g else float('nan'),
            'precision': round(c / n, 2) if n else float('nan'),
            '빠뜨림': int(df['빠뜨림'].sum()), '없는물체': int(df['없는물체'].sum()),
            '구역틀림': int(df['구역틀림'].sum())}

summary = pd.DataFrame([
    micro(df_col, 'A. 색 (json, 512px)'),
    micro(df_num[df_num['프롬프트'] == 'number'], 'B. 숫자 (number, 512px)'),
    micro(df_num[df_num['프롬프트'] == 'number_read'], 'C. 숫자 (number_read, 512px)'),
])
display(summary)

print('\n읽는 법:')
print('  A > B ≈ C  →  색 단서가 쉬워서 A의 점수가 높았다. 숫자 읽기 자체가 병목.')
print('  A > B > C  →  숫자를 읽지 못하는 문제 + 규칙 적용 문제가 겹쳐 있다.')
print('  A ≈ B ≈ C  →  구분 단서와 무관하게 계획 능력이 동일하다 (가능성 낮음).')
print('  C > B      →  숫자는 읽지만 규칙 적용에서 틀린다.')
print()
print('※ 씬 %d개 기준입니다. 보고서에는 SEEDS 를 30개로 늘려서 재세요.' % len(SEEDS))

---
## SECTION 3 | 해상도 실험 — 256px vs 512px

숫자 조건에서 **이미지 해상도**가 VLM 성능에 얼마나 영향을 주는지 확인합니다.

- 색은 한 픽셀만 봐도 알 수 있다 → 해상도에 둔감
- 숫자는 글자 모양을 읽어야 한다 → 해상도가 낮으면 읽을 수 없다

이 차이가 실측으로 확인되면, "숫자를 읽는 것이 이 과제의 병목"이라는 주장의 직접적 근거가 됩니다.

In [ ]:
# 3-1. 숫자 조건을 256px / 512px 로 나란히 채점
rows_res = []
for px in [256, 512]:
    vlm.prompt_style = 'number'
    for seed in SEEDS:
        env_num.reset(layout='messy', seed=seed)
        g = RulePlanner(env_num).plan()
        p = parse_plan(vlm.raw_response(
            env_num.render('agentview', px, px), '테이블을 정리하시오'), allow_default_zone=(vlm.prompt_style == 'number_read'))
        a = plan_accuracy(p, g)
        rows_res.append({'해상도': f'{px}px', 'seed': seed,
                         'recall': round(a['recall'], 2), 'precision': round(a['precision'], 2),
                         '맞춘수': a['correct'], '정답수': a['n_gold'], '예측수': a['n_pred']})

df_res = pd.DataFrame(rows_res)

def micro_res(df, label):
    c, g, n = int(df['맞춘수'].sum()), int(df['정답수'].sum()), int(df['예측수'].sum())
    return {'조건': label, 'recall': round(c/g, 2) if g else 0, 'precision': round(c/n, 2) if n else 0}

display(pd.DataFrame([micro_res(df_res[df_res['해상도']==r], f'숫자 ({r})') for r in ['256px', '512px']]))

print('\n해상도를 반으로 줄이면 점수가 얼마나 떨어지나요?')
print('많이 떨어진다면 → VLM이 숫자를 읽는 것이 병목입니다.')
print('비슷하다면 → 어차피 숫자를 못 읽고 있으므로 해상도와 무관합니다.')

---
## SECTION 4 | 2단계 — VLA 정책으로 숫자 큐브 옮기기

1단계(VLM)가 계획을 세우면, 2단계(VLA)가 실제로 물체를 집어 옮깁니다.

여기서는 먼저 **규칙 플래너(정답)** + **ScriptedSortingPolicy(teacher)**로 돌려
숫자 조건에서 제어 자체가 잘 되는지 확인합니다.

색 조건과 비교해서 **제어 성능이 비슷하면** → 1단계(VLM)만의 문제라고 말할 수 있습니다.

In [ ]:
# 4-1. 숫자 조건 — 규칙 플래너 + teacher 정책으로 rollout
from vlapipe.policy import ScriptedSortingPolicy
import imageio.v2 as imageio
from IPython.display import Image as IPyImage, display as ipy_display

env_num.reset(layout='messy', seed=3)
policy = ScriptedSortingPolicy(env_num)
plan = RulePlanner(env_num).plan('테이블을 정리하시오')

frames, log = [], []
total_steps = 0
for st in plan:
    policy.set_subtask(st); n = 0
    while not policy.done and n < 500:
        env_num.step(policy.select_action()); n += 1
        if n % 3 == 0:
            frames.append(env_num.render('frontview', 300, 400))
    total_steps += n
    ok = env_num.zone_of(st['object']) == st['target_zone']
    log.append({'물체': st['object'], '스텝': n, '성공': ok, '실패사유': policy.failure})

df_log = pd.DataFrame(log)
display(df_log)
print(f'\n정리 완료: {env_num.is_tidy()} | 총 스텝: {total_steps}')
print(f'남은 물체: {env_num.misplaced_objects()}')

imageio.mimsave('outputs/section4_number.gif', frames[::2], fps=14, loop=0)
ipy_display(IPyImage(filename='outputs/section4_number.gif'))

In [ ]:
# 4-2. 여러 시드에서 숫자 조건의 teacher 성공률
results_teacher = []
for seed in SEEDS:
    env_num.reset(layout='messy', seed=seed)
    pol = ScriptedSortingPolicy(env_num)
    pln = RulePlanner(env_num).plan()
    ok_count, fail_count = 0, 0
    steps = 0
    for st in pln:
        pol.set_subtask(st); n = 0
        while not pol.done and n < 500:
            env_num.step(pol.select_action()); n += 1
        steps += n
        if env_num.zone_of(st['object']) == st['target_zone']:
            ok_count += 1
        else:
            fail_count += 1
    results_teacher.append({
        'seed': seed, '계획수': len(pln), '성공': ok_count, '실패': fail_count,
        '스텝': steps, '정리완료': env_num.is_tidy()})

df_teacher = pd.DataFrame(results_teacher)
display(df_teacher)

total_ok = df_teacher['성공'].sum()
total_all = (df_teacher['성공'] + df_teacher['실패']).sum()
print(f'\nteacher 운반 성공률: {total_ok}/{total_all} ({total_ok/total_all*100:.0f}%)')
print(f'씬 정리 완료: {df_teacher["정리완료"].sum()}/{len(df_teacher)}')
print('\n이 결과가 색 조건(12회차)과 비슷하면 → 제어는 문제없고 1단계(VLM)만 병목.')

---
## SECTION 5 | 엔드투엔드 — VLM 계획 + VLA 실행

지금까지는 1단계(VLM)와 2단계(VLA)를 따로따로 봤습니다.
이제 **진짜 파이프라인**: VLM이 이미지를 보고 계획을 세우고, VLA가 실행합니다.

```
이미지(숫자 큐브) + "테이블을 정리하시오"
  │
  ├─ [1단계 VLM]  SmolVLM2 : 숫자를 읽고 이동 계획 생성
  │                         "Pick up the cube number 7 and place it in the left storage zone."
  └─ [2단계 VLA]  Teacher  : 지시문에 따라 관절 제어
```

In [ ]:
# 5-1. 숫자 조건 엔드투엔드 (VLM 플래너 + teacher 정책)
!python scripts/run_pipeline.py --objects number --layout messy --seed 3 --passes 2 --planner vlm

In [ ]:
# 5-2. 같은 시드에서 규칙 플래너(정답)로도 돌려 비교
!python scripts/run_pipeline.py --objects number --layout messy --seed 3 --passes 2 --planner rule

In [ ]:
# 5-3. 여러 시드에서 VLM 플래너 엔드투엔드 성능
vlm.prompt_style = 'number'

results_e2e = []
for seed in SEEDS:
    env_num.reset(layout='messy', seed=seed)
    img_vlm = env_num.render('agentview', VLM_PX, VLM_PX)
    raw_plan = vlm.raw_response(img_vlm, '테이블을 정리하시오')
    vlm_plan = parse_plan(raw_plan, allow_default_zone=(vlm.prompt_style == 'number_read'))
    gold_plan = RulePlanner(env_num).plan()
    acc_e2e = plan_accuracy(vlm_plan, gold_plan)

    pol = ScriptedSortingPolicy(env_num)
    ok_count, fail_count, steps = 0, 0, 0
    for st in vlm_plan:
        pol.set_subtask(st); n = 0
        while not pol.done and n < 500:
            env_num.step(pol.select_action()); n += 1
        steps += n
        if env_num.zone_of(st['object']) == st['target_zone']:
            ok_count += 1
        else:
            fail_count += 1

    results_e2e.append({
        'seed': seed, 'VLM계획수': len(vlm_plan), '정답수': len(gold_plan),
        'plan_recall': round(acc_e2e['recall'], 2),
        '운반성공': ok_count, '운반실패': fail_count,
        '스텝': steps, '정리완료': env_num.is_tidy(),
        '남은물체': len(env_num.misplaced_objects())})

df_e2e = pd.DataFrame(results_e2e)
display(df_e2e)

print(f'\n평균 plan recall: {df_e2e["plan_recall"].mean():.2f}')
print(f'씬 정리 완료: {df_e2e["정리완료"].sum()}/{len(df_e2e)}')

---
## SECTION 6 | 오류 분석 — 어디서 틀리는가

엔드투엔드에서 실패했을 때 **어디서 틀렸는지**를 분리합니다.

1. **VLM이 틀린 것**: 없는 물체를 옮기라고 함(환각), 빠뜨림, 구역 틀림
2. **VLA가 틀린 것**: 물체를 못 잡음, 이송 중 떨어뜨림
3. **해상도 때문인 것**: 512px→256px 로 바꿨을 때 VLM 점수가 떨어지면 지각 문제

이 분리가 되어야 "무엇을 고쳐야 하는가"에 답할 수 있습니다.

In [ ]:
# 6-1. VLM 오류 유형 집계
print('=== VLM 오류 유형 (숫자 조건, number 프롬프트) ===')
num_df = df_num[df_num['프롬프트'] == 'number']
print(f'  빠뜨림 (정답에 있는데 말하지 않음) : {num_df["빠뜨림"].sum()}')
print(f'  없는물체 (옮길 필요 없는 것을 옮기라 함) : {num_df["없는물체"].sum()}')
print(f'  구역틀림 (물체는 맞고 목적지가 틀림) : {num_df["구역틀림"].sum()}')
print()

print('=== VLM 오류 유형 (색 조건, json 프롬프트) ===')
print(f'  빠뜨림 : {df_col["빠뜨림"].sum()}')
print(f'  없는물체 : {df_col["없는물체"].sum()}')
print(f'  구역틀림 : {df_col["구역틀림"].sum()}')
print()

print('숫자 조건의 "없는물체"가 많다면 → VLM이 숫자를 아무거나 찍고 있다 (읽지 않음)')
print('숫자 조건의 "구역틀림"이 많다면 → 숫자는 읽지만 1~5/6~10 규칙을 모른다')
print('숫자 조건의 "빠뜨림"이 많다면 → 숫자를 보는 것 자체가 안 된다')

In [ ]:
# 6-2. VLM이 특정 숫자를 잘/못 읽는지 분석
# VLM 출력에서 언급된 숫자를 추출하고, 정답과 비교
import re

mentioned_numbers = []  # VLM이 언급한 숫자들
correct_numbers = []    # 정답에 있는 숫자들

vlm.prompt_style = 'number'
for seed in SEEDS:
    env_num.reset(layout='messy', seed=seed)
    g = RulePlanner(env_num).plan()
    p = parse_plan(vlm.raw_response(
        env_num.render('agentview', VLM_PX, VLM_PX), '테이블을 정리하시오'), allow_default_zone=(vlm.prompt_style == 'number_read'))

    # 정답에서 숫자 추출
    for st in g:
        m = re.search(r'(\d+)', st['object'])
        if m: correct_numbers.append(int(m.group(1)))

    # VLM 예측에서 숫자 추출
    for st in p:
        m = re.search(r'(\d+)', st['object'])
        if m: mentioned_numbers.append(int(m.group(1)))

from collections import Counter
pred_cnt = Counter(mentioned_numbers)
gold_cnt = Counter(correct_numbers)

print('VLM이 언급한 숫자 빈도:', dict(sorted(pred_cnt.items())))
print('정답에 있는 숫자 빈도:', dict(sorted(gold_cnt.items())))
print()

# 0~10 범위 밖의 숫자를 언급했다면 완전히 엉뚱한 것
out_of_range = [n for n in mentioned_numbers if n < 1 or n > 10]
if out_of_range:
    print(f'범위 밖 숫자 언급: {out_of_range} → VLM이 숫자를 읽지 않고 있다')
else:
    print('범위 밖 숫자 없음 (1~10 안에서 답함)')

print('\n특정 숫자 쌍(6/9, 1/7)에서 헷갈리는지 확인하세요:')
print('  6과 9를 둘 다 많이 언급 → 글자 모양 혼동 가능성')
print('  아무 숫자나 골고루 찍음 → 아예 읽지 않고 있음')

In [ ]:
# 6-3. 최종 비교 요약표
print('=' * 70)
print('  최종 비교: 색 vs 숫자 (같은 시드 · 같은 layout · 같은 모델)')
print('=' * 70)
display(summary)  # 2-6 에서 만든 표
print()
print('--- teacher 제어 (규칙 플래너 기준) ---')
display(df_teacher[['seed', '성공', '실패', '정리완료']])
print()
print('--- 엔드투엔드 (VLM + teacher) ---')
display(df_e2e[['seed', 'plan_recall', '운반성공', '운반실패', '정리완료']])
print()
print('--- 해상도 영향 ---')
display(pd.DataFrame([micro_res(df_res[df_res['해상도']==r], f'숫자 ({r})') for r in ['256px', '512px']]))
print()
print('위 표들로 다음을 판단하세요:')
print('  1) teacher 제어가 색 조건과 비슷한가? → 비슷하면 2단계는 문제 없음')
print('  2) VLM recall 이 색 > 숫자인가? → 그렇다면 1단계의 "지각" 이 병목')
print('  3) 해상도를 올리면 recall 이 오르는가? → 그렇다면 이미지 해상도가 핵심')

---
## SECTION 7 | 데이터 수집 — 숫자 조건 텔레옵 녹화 + 역재생

SmolVLA 를 숫자 조건에서 파인튜닝하려면 **숫자 큐브를 옮기는 데모 데이터**가 필요합니다.

12회차와 같은 방법: 정돈된 씬을 어지르는 쪽을 게임패드로 녹화하고, 역재생으로 정돈 데모를 만듭니다.

**녹화 명령** (윈도우 로컬에서):
```powershell
.\.\.venv\Scripts\python.exe scripts\teleop_record.py --objects number --input keyboard
```

In [ ]:
# 7-1. 역재생 변환 + 검증
!python scripts/build_dataset.py

In [ ]:
# 7-2. 역재생 결과 눈으로 확인
from vlapipe.episode import Episode
import glob

revs = sorted(glob.glob('data/episodes_reversed/*.hdf5'))
if revs:
    ep = Episode(revs[0])
    print(ep.path.name, '|', len(ep), 'frames | task:', ep.task)
    idx = np.linspace(0, len(ep) - 1, 6).astype(int)
    fig, ax = plt.subplots(1, 6, figsize=(18, 3))
    for a, i in zip(ax, idx):
        a.imshow(ep.obs['agentview_image'][i]); a.axis('off'); a.set_title(f't={i}', fontsize=8)
    plt.suptitle('Reversed tidying demo (left=start, right=end)'); plt.tight_layout(); plt.show()
    imageio.mimsave('outputs/reversed_number.gif', list(ep.obs['agentview_image'][::3]), fps=12, loop=0)
    ipy_display(IPyImage(filename='outputs/reversed_number.gif'))
else:
    print('아직 녹화 데이터가 없습니다.')
    print('윈도우에서 먼저 실행하세요:')
    print('  .\\.\\.venv\\Scripts\\python.exe scripts\\teleop_record.py --objects number --input keyboard')

---
## SECTION 8 | SmolVLA 파인튜닝 (데이터를 모은 뒤)

> 에피소드 20~50개가 모인 뒤에 실행하세요.

In [ ]:
# 8-1. LeRobot 설치
!pip install -q 'lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git@9e5ee9859480e425a97a6836bb3fee479539beb5'

In [ ]:
# 8-2. LeRobot 데이터셋으로 변환
!python scripts/build_dataset.py --lerobot --repo-id dshs2026/vla_number_sorting

In [ ]:
# 8-3. SmolVLA 파인튜닝
# Use train-only data prepared with seed-based holdout splitting.
!python scripts/train_vla.py \
  --dataset-root=data/lerobot --repo-id=dshs2026/vla_number_sorting \
  --batch-size=8 --steps=2000 \
  --out=outputs/smolvla_number_sorting --device=cuda


In [ ]:
# 8-4. 학습된 SmolVLA 로 숫자 조건 파이프라인 실행
!python scripts/run_pipeline.py --objects number --policy smolvla \
  --checkpoint outputs/smolvla_number_sorting/checkpoints/last/pretrained_model \
  --layout messy --seed 3

---
## SECTION 9 | 미니과제

### 필수

#### 1. 비교표 채우기

SECTION 2~3 을 돌려 아래 표를 **실측값**으로 채우세요. 추측한 값을 적지 마세요.

| 조건 | 프롬프트 | 입력 px | 씬 수 | recall | precision | 빠뜨림 | 없는물체 | 구역틀림 |
|---|---|---|---|---|---|---|---|---|
| 색 | `json` | 512 | | | | | | |
| 숫자 | `number` | 512 | | | | | | |
| 숫자 | `number_read` | 512 | | | | | | |
| 숫자 | `number` | 256 | | | | | | |

표 아래에 **2~3문장**: 어느 쪽 문제였나? 무엇을 근거로 그렇게 보았나?

#### 2. 제어 비교

SECTION 4 의 teacher 결과를 12회차(색 조건)와 비교하세요.
- 운반 성공률이 비슷한가?
- 실패 사유(`grasp_failed`, `dropped_during_carry` 등)가 다른가?
- 비슷하다면 "제어 문제가 아니라 지각 문제"라고 말할 수 있습니다.

#### 3. 엔드투엔드 실패 분석

SECTION 5 에서 VLM + teacher 로 돌렸을 때:
- 정리 완료한 씬이 몇 개인가?
- 실패 원인이 1단계(VLM 계획 오류)인지 2단계(VLA 실행 실패)인지 적으세요.

### 생각할 질문

1. **숫자를 더 잘 읽게 하려면?** 세 가지 방법의 장단점을 비교하세요:
   - 이미지 해상도를 올린다 (512 → 768)
   - 숫자 텍스처를 더 크고 또렷하게 만든다
   - VLM 을 파인튜닝한다

2. **색 조건에서 높은 점수가 진짜 능력인가?**
   색은 한 픽셀만 봐도 구분되니까, VLM 이 이미지를 "제대로 이해"한 것이 아닐 수 있습니다.
   이 주장을 뒷받침하는 근거를 여러분의 실험 결과에서 찾으세요.

3. **실제 로봇에서 숫자 읽기는 더 어려울까?**
   시뮬레이션은 조명이 균일하고 숫자가 깨끗합니다. 실제로는?
   (힌트: 그림자, 각도, 빛 반사, 마모)

## Before comparing architectures: train two appropriately labelled policies

SECTION 8 demonstrates a training connection; it does not train both comparison systems.
Use the same original demonstrations and seed-based train/holdout split for both systems.
Single VLA needs whole-scene instructions; hierarchical VLA needs matched per-object action segments.
Changing only the task string of a whole-scene demonstration does not create valid subtask supervision.

`collect_scripted.py --task-level subtask` records successful object segments.
`--task-level scene` records whole-scene demonstrations. Use identical source seeds for both.
Run `prepare_vla_training.py` for each system with identical holdout seeds before conversion.
Set dataset paths below, then deliberately enable RUN_COMPARISON_TRAINING.
A short training smoke run checks software only, not research performance.


In [ ]:
# Optional: train two policies on separately labelled, train-only datasets
import subprocess, sys
RUN_COMPARISON_TRAINING = False
TRAIN_DATA = {
    'hier': {'root': Path('data/prepared_hier/lerobot'), 'repo': 'dshs2026/vla_hier',
             'output': Path('outputs/smolvla_number_sorting')},
    'single': {'root': Path('data/prepared_single/lerobot'), 'repo': 'dshs2026/vla_single',
               'output': Path('outputs/smolvla_number_sorting_single')},
}
if RUN_COMPARISON_TRAINING:
    for system, spec in TRAIN_DATA.items():
        assert (spec['root'] / 'meta/info.json').is_file(), f'{system}: convert the labelled training dataset first'
        assert not spec['output'].exists(), f"Existing output: {spec['output']}; use a new output path"
        subprocess.run([sys.executable, 'scripts/train_vla.py',
            f"--repo-id={spec['repo']}", f"--dataset-root={spec['root']}",
            '--batch-size=8', '--steps=2000', f"--out={spec['output']}",
            '--device=cuda'], check=True)
else:
    print('Training skipped. Prepare matching labelled datasets and held-out seeds first.')


---
## SECTION 10 | 본 실험 — 단일 구조(VLA) vs 계층 구조(VLM→VLA)

**가설**: 계층 구조(VLM-VLA) 택배 정리 로봇은 단일 구조(VLA)보다
작업시간이 짧고, Hit-rate·F1이 높으며, 행동 MSE가 작다.
물체가 많거나 지시문이 모호할수록 차이가 커진다.

| 독립변수 | 값 |
|---|---|
| 구조 | `single_vla` (지시문 → SmolVLA 바로) / `hier_vla` (지시문 → SmolVLM2 계획 → SmolVLA) |
| 물체 수 | 10 / 20 / 30 |
| 지시문 명확도 | `explicit` / `vague` / `very_vague` |
| 반복 | 조건당 10 시드 |

| 종속변수 | 정의 |
|---|---|
| Hit-rate | 종료 시점에 올바른 구역에 있는 물체 비율 (전체 n개 기준) |
| 성공률 | 씬 전체가 정돈된 시행의 비율 |
| F1 | 처음에 잘못 놓인 물체 중 올바르게 옮긴 것(TP), 못 옮긴 것(FN), 멀쩡한 걸 망친 것(FP) 기준 |
| 작업시간 | 시뮬레이션 스텝/제어주기 + VLM 추론시간 (계층 구조의 지연도 포함) |
| 행동 MSE | 홀드아웃 시연 데이터에서 정책 행동 vs 정답 행동 (10-8) |

`hier_teacher`(VLM + 스크립트 제어기)는 **참고용 상한선**입니다. 가설 비교 대상이 아닙니다.

> **먼저 읽기** — 이 섹션에서 사람이 확인해야 하는 곳은 `# [확인]` 주석이 붙어 있습니다.
### Interpretation limits
The hierarchical VLM prompt always supplies the sorting rule, even for vague user instructions.
If the single VLA has not learned the same rule through equivalent supervision, an apparent advantage
mixes architecture with extra information. Report this as a confound; test a rule-explicit single VLA
and a matched-rule hierarchical prompt before attributing robustness to architecture alone.
Use 20/30-object settings only after reachability, non-overlap and storage capacity checks.


In [ ]:
# 10-1. 실험 설정
import time, json, math
import numpy as np, pandas as pd
from pathlib import Path

N_OBJECTS_LIST = [10, 20, 30]
N_TRIALS       = 10                         # 조건당 반복 횟수 (스모크 테스트에서는 2로 줄여도 됨)
TRIAL_SEEDS    = list(range(100, 100 + N_TRIALS))   # 5~13 의 SEEDS 와 겹치지 않게
MAX_STEPS_PER_OBJECT = 500                  # 물체 1개당 스텝 상한 (두 구조에 동일하게 적용)
PASSES   = 2                                # 계층 구조: 남은 물체가 있으면 다시 계획 (run_pipeline --passes 2 와 동일)
VLM_PX   = 512

# 지시문 3단계. VLA 가 영어 지시문으로 학습되었다고 가정하고 영어로 둠.   # [확인]
INSTRUCTIONS = {
    'explicit':   lambda n: f'Tidy the table: put cubes 1-{n//2} in the right zone and cubes {n//2+1}-{n} in the left zone.',
    'vague':      lambda n: 'Tidy up the table.',
    'very_vague': lambda n: 'Clean up.',
}

# 체크포인트. 단일 구조는 "전체 정리" 지시문으로 학습한 별도 체크포인트가 있어야 공정함.   # [확인]
CKPT_HIER   = 'outputs/smolvla_number_sorting/checkpoints/last/pretrained_model'
CKPT_SINGLE = 'outputs/smolvla_number_sorting_single/checkpoints/last/pretrained_model'
INCLUDE_TEACHER_REF = True                  # hier_teacher 상한선 같이 돌리기 (싸다)

CONTROL_HZ = getattr(C, 'CONTROL_FREQ', getattr(C, 'CONTROL_HZ', None))
if CONTROL_HZ is None:
    CONTROL_HZ = 20
    print('⚠ config 에서 제어 주기를 못 찾아 20 Hz 로 가정했습니다. 실제 값으로 바꾸세요 (CONTROL_HZ).')

RESULT_DIR = (TEAM / 'results') if IN_COLAB else (PKG / 'outputs')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_JSONL = RESULT_DIR / 'exp_results_v2.jsonl'   # 시행마다 한 줄씩 저장 → 런타임이 끊겨도 이어서 가능
print('결과 파일:', RESULT_JSONL, '| 제어주기:', CONTROL_HZ, 'Hz')
# Resume only when the complete protocol is unchanged. Use a new results path otherwise.
import hashlib
SOURCE_HASHES = {str(p): hashlib.sha256(p.read_bytes()).hexdigest()
                 for p in (Path('vlapipe/planner.py'), Path('vlapipe/numbered.py'),
                           Path('vlapipe/policy.py'), Path('vlapipe/scene.py'))}
PROTOCOL = {'version': 2, 'objects': N_OBJECTS_LIST, 'seeds': TRIAL_SEEDS,
            'max_steps_per_object': MAX_STEPS_PER_OBJECT, 'passes': PASSES,
            'vlm_px': VLM_PX, 'ckpt_hier': CKPT_HIER, 'ckpt_single': CKPT_SINGLE,
            'include_teacher': INCLUDE_TEACHER_REF, 'source_hashes': SOURCE_HASHES,
            'instructions': {k: [f(n) for n in N_OBJECTS_LIST] for k, f in INSTRUCTIONS.items()},
            'adapter': 'saved_processors_state16_action7_v2',
            'vlm_model': 'HuggingFaceTB/SmolVLM2-500M-Video-Instruct',
            'camera_names': list(C.ALL_CAMERAS), 'image_size': C.IMAGE_SIZE}
PROTOCOL_PATH = RESULT_JSONL.with_suffix('.protocol.json')
if PROTOCOL_PATH.exists():
    assert json.loads(PROTOCOL_PATH.read_text()) == PROTOCOL, 'Protocol changed: choose a new RESULT_JSONL.'
else:
    assert not RESULT_JSONL.exists(), 'Existing results lack protocol metadata; use a new output path.'
    PROTOCOL_PATH.write_text(json.dumps(PROTOCOL, indent=2))


In [ ]:
# 10-2. 물체 수를 바꾸는 환경 생성기
from vlapipe.env import SortingEnv
from vlapipe import config as C, numbered
from vlapipe.planner import RulePlanner, VLMPlanner, parse_plan, PROMPTS as PLANNER_PROMPTS
from vlapipe.policy import ScriptedSortingPolicy

def make_env(n):
    """숫자 큐브 n개 환경. config 를 바꾼 뒤 numbered 를 다시 켠다."""
    numbered.deactivate()
    numbered.activate(count=n)
    env = SortingEnv(layout='messy', seed=TRIAL_SEEDS[0], image_size=C.IMAGE_SIZE, cameras=C.ALL_CAMERAS)
    env.reset(layout='messy', seed=TRIAL_SEEDS[0])
    got = len(C.OBJECT_NAMES)
    assert got == n, (
        f'물체가 {got}개로 만들어졌습니다 (요청 {n}개). '
        'vlapipe/config.py, vlapipe/numbered.py 에서 NUMBER_COUNT 가 어디서 쓰이는지 확인하세요 '
        '(import 시점에 고정되면 numbered.activate() 안에서 다시 읽도록 고쳐야 합니다).')
    mis = env.misplaced_objects()
    assert len(mis) > 0, '씬이 이미 정돈되어 있습니다. messy 배치가 n개 큐브를 지원하는지 확인하세요.'
    return env

# [확인] VLM 프롬프트 템플릿에 1~5/6~10 규칙이 박혀 있으면 n=20,30 에서 틀린 규칙을 쓰게 됩니다.
print('--- VLMPlanner "number" 프롬프트 ---')
print(PLANNER_PROMPTS['number'])

In [ ]:
# 10-3. 실행기(executor) 어댑터 — 두 구조가 같은 인터페이스를 쓰도록 맞춘다
def misplaced_or_airborne(env):
    bad = set(env.misplaced_objects())
    for name in C.OBJECT_NAMES:
        resting_z = C.TABLE_TOP_Z + C.half_size(C.OBJECT_BY_NAME[name])[2]
        if abs(env.object_pos(name)[2] - resting_z) > 0.025:
            bad.add(name)
    return bad


def gripper_released(env):
    # Actual finger state, not the most recently requested open command.
    return float(np.min(env.data.qpos[env._finger_adr])) > 0.035


def scene_completed(env):
    return not misplaced_or_airborne(env) and gripper_released(env)


def released_and_placed(env, name, zone):
    resting_z = C.TABLE_TOP_Z + C.half_size(C.OBJECT_BY_NAME[name])[2]
    return (env.zone_of(name) == zone and abs(env.object_pos(name)[2] - resting_z) <= 0.025
            and gripper_released(env))


class TeacherExecutor:
    """스크립트 teacher (상한선). 지시문만으로는 계획을 못 세우므로 단일 구조에는 쓸 수 없다."""
    name = 'teacher'
    def reset(self, env):
        self.env = env
        self.policy = ScriptedSortingPolicy(env)
    def run_subtask(self, st, max_steps):
        self.policy.set_subtask(st); n = 0
        while not self.policy.done and n < max_steps:
            self.env.step(self.policy.select_action()); n += 1
        return n
    def run_instruction(self, text, max_steps):
        raise NotImplementedError('teacher 는 지시문만으로 계획을 못 세웁니다')


class LearnedVLAExecutor:
    """The same checked adapter is used by CLI and notebook (saved processors included)."""
    def __init__(self, ckpt, name):
        from vlapipe.policy import SmolVLAPolicy
        if not Path(ckpt).is_dir():
            raise FileNotFoundError(f'{name}: checkpoint missing: {ckpt}. Train and validate this system first.')
        self.name = name
        self.adapter = SmolVLAPolicy(ckpt)

    def reset(self, env):
        self.env = env
        self.adapter.reset()

    def _loop(self, text, max_steps, done_fn, check_every=10):
        self.adapter.reset(); n = ok_streak = 0
        while n < max_steps:
            obs = self.env.get_obs(); obs['task'] = text
            self.env.step(self.adapter.select_action(obs)); n += 1
            if n % check_every == 0:
                ok_streak = ok_streak + 1 if done_fn() else 0
                if ok_streak >= 2: break  # actual release + supported height, twice 10 steps apart
        return n

    def run_subtask(self, st, max_steps):
        return self._loop(st['instruction'], max_steps,
                          lambda: released_and_placed(self.env, st['object'], st['target_zone']))

    def run_instruction(self, text, max_steps):
        return self._loop(text, max_steps, lambda: scene_completed(self.env))


In [ ]:
# 10-4. 시행 1회 실행 + 지표 계산
def run_trial(env, mode, executor, n, prompt_key, seed, vlm):
    env.reset(layout='messy', seed=seed)
    wall_start = time.perf_counter()
    executor.reset(env)
    init = misplaced_or_airborne(env)
    instr = INSTRUCTIONS[prompt_key](n)
    steps, vlm_s, n_plan, errors = 0, 0.0, 0, 0
    budget = MAX_STEPS_PER_OBJECT * max(len(init), 1)

    if mode == 'hier':
        vlm.prompt_style = 'number'
        for _ in range(PASSES):
            if scene_completed(env) or steps >= budget: break
            t0 = time.perf_counter()
            plan = parse_plan(vlm.raw_response(env.render('agentview', VLM_PX, VLM_PX), instr), allow_default_zone=(vlm.prompt_style == 'number_read'))
            vlm_s += time.perf_counter() - t0
            n_plan += len(plan)
            for st in plan:
                if steps >= budget: break
                try:
                    steps += executor.run_subtask(st, min(MAX_STEPS_PER_OBJECT, budget - steps))
                except Exception as e:           # 환각 물체 등으로 실행기가 터진 경우도 실패로 센다
                    # Partial execution must also consume the global budget.
                    # Abort this trial rather than hide unknown consumed steps.
                    raise RuntimeError(f"executor failed: n={n}, seed={seed}, subtask={st}") from e
    else:
        steps = executor.run_instruction(instr, budget)

    final = misplaced_or_airborne(env)
    tp, fn, fp = len(init - final), len(init & final), len(final - init)
    den = 2 * tp + fp + fn
    sim_s = steps / CONTROL_HZ
    return {
        'n_objects': n, 'prompt': prompt_key, 'seed': seed,
        'n_init_misplaced': len(init), 'n_final_misplaced': len(final),
        'hit_rate': 1 - len(final) / n, 'success': int(scene_completed(env)),
        'tp': tp, 'fp': fp, 'fn': fn, 'f1': (2 * tp / den) if den else float('nan'),
        'steps': steps, 'sim_s': sim_s, 'vlm_s': vlm_s,
        'task_time_s': sim_s + vlm_s,  # hybrid proxy; not actual wall latency
        'wall_s': time.perf_counter() - wall_start,
        'n_plan': n_plan, 'exec_errors': errors, 'step_budget': budget,
    }

In [ ]:
# 10-5. 실행기 준비 + 스모크 테스트 (n=10, vague, 시드 1개씩 → 구조별로 잘 도는지만 확인)
vlm = VLMPlanner('HuggingFaceTB/SmolVLM2-500M-Video-Instruct', prompt_style='number') if 'vlm' not in dir() else vlm

if CKPT_HIER == CKPT_SINGLE:
    print('⚠ 두 구조가 같은 체크포인트입니다. 단일 구조는 "전체 정리" 지시문으로 학습한 별도 체크포인트가 있어야 공정합니다.')

# Run training separately for whole-task and subtask-labelled datasets.
for ckpt in (CKPT_SINGLE, CKPT_HIER):
    assert Path(ckpt).is_dir(), f'Missing checkpoint: {ckpt}; this notebook is not evidence of completed training.'
SYSTEMS = {}
SYSTEMS['single_vla'] = ('single', LearnedVLAExecutor(CKPT_SINGLE, 'single_vla'))
SYSTEMS['hier_vla']   = ('hier',   LearnedVLAExecutor(CKPT_HIER,   'hier_vla'))
if INCLUDE_TEACHER_REF:
    SYSTEMS['hier_teacher'] = ('hier', TeacherExecutor())

env10 = make_env(10)
for name, (mode, ex) in SYSTEMS.items():
    t0 = time.perf_counter()
    r = run_trial(env10, mode, ex, 10, 'vague', TRIAL_SEEDS[0], vlm)
    print(f"{name:13s} hit={r['hit_rate']:.2f} f1={r['f1']:.2f} steps={r['steps']:5d} "
          f"time={r['task_time_s']:.1f}s err={r['exec_errors']} (실측 {time.perf_counter()-t0:.0f}s)")
print('\n※ 실측 시간 × (3 물체수 × 3 프롬프트 × 3 시스템 × 시드수) 로 전체 소요를 어림해 보세요.')
env10.close()


In [ ]:
# 10-6. 본 실험 실행 (이어하기 지원: 끊겨도 같은 셀을 다시 돌리면 남은 것만 한다)
DONE = set()
if RESULT_JSONL.exists():
    for line in RESULT_JSONL.read_text(encoding='utf-8').splitlines():
        r = json.loads(line); DONE.add((r['system'], r['n_objects'], r['prompt'], r['seed']))
print('이미 끝난 시행:', len(DONE))

total = len(N_OBJECTS_LIST) * len(INSTRUCTIONS) * len(SYSTEMS) * len(TRIAL_SEEDS)
t_start = time.perf_counter()
for n in N_OBJECTS_LIST:
    env_n = make_env(n)
    for prompt_key in INSTRUCTIONS:
        for name, (mode, ex) in SYSTEMS.items():
            for seed in TRIAL_SEEDS:
                if (name, n, prompt_key, seed) in DONE: continue
                row = run_trial(env_n, mode, ex, n, prompt_key, seed, vlm)
                row['system'] = name
                with open(RESULT_JSONL, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(row, ensure_ascii=False) + '\n')
                DONE.add((name, n, prompt_key, seed))
    env_n.close()
    print(f'n={n} 완료 | 누적 {len(DONE)}/{total} | {(time.perf_counter()-t_start)/60:.1f} 분')

In [ ]:
# 10-7. 결과 분석 — "B가 기존보다 몇 % 나아졌다"
df = pd.read_json(RESULT_JSONL, lines=True)
agg = (df.groupby(['n_objects', 'prompt', 'system'])
         .agg(시행수=('seed', 'count'),
              hit=('hit_rate', 'mean'), hit_sd=('hit_rate', 'std'),
              성공률=('success', 'mean'),
              f1=('f1', 'mean'), f1_sd=('f1', 'std'),
              시간_s=('wall_s', 'mean'), 시간_sd=('wall_s', 'std'))
         .round(3).reset_index())
display(agg)

# 단일 vs 계층 비교 (상대 개선 % 와 절대 차이 %p, paired Wilcoxon p값)
try:
    from scipy.stats import wilcoxon
except ImportError:
    wilcoxon = None

def rel(new, old, lower_better=False):
    if old == 0 or pd.isna(old): return float('nan')
    d = (old - new) / old if lower_better else (new - old) / old
    return round(100 * d, 1)

rows = []
for (n, p), g in df.groupby(['n_objects', 'prompt']):
    s, h = g[g.system == 'single_vla'], g[g.system == 'hier_vla']
    if s.empty or h.empty: continue
    row = {'물체수': n, '지시문': p, '시행수(단일/계층)': f'{len(s)}/{len(h)}'}
    for col, label, lower in [('hit_rate', 'Hit-rate', False), ('f1', 'F1', False), ('wall_s', 'Wall time', True)]:
        paired = h[['seed', col]].merge(s[['seed', col]], on='seed', suffixes=('_h', '_s')).dropna()
        sm, hm = paired[col + '_s'].mean(), paired[col + '_h'].mean()
        row[f'{label} 단일'] = round(sm, 3); row[f'{label} 계층'] = round(hm, 3)
        row[f'{label} 상대개선%'] = rel(hm, sm, lower)
        if col != 'task_time_s': row[f'{label} 차이%p'] = round(100 * (hm - sm), 1)
        paired = h[['seed', col]].merge(s[['seed', col]], on='seed', suffixes=('_h', '_s')).dropna()
        if wilcoxon is not None and len(paired) > 1:
            delta = paired[col + '_h'] - paired[col + '_s']
            row[f'{label} paired_p'] = 1.0 if np.allclose(delta, 0) else round(wilcoxon(delta).pvalue, 4)
    rows.append(row)
cmp_df = pd.DataFrame(rows)
display(cmp_df)

print('읽는 법: 상대개선% > 0 이면 계층 구조가 더 좋음 (작업시간은 짧을수록 좋음 → 양수가 개선).')
print('Paired seed comparisons are exploratory. Correct for multiple tests and report uncertainty.')
print('기준선(단일)이 0 에 가까우면 상대개선%가 폭발하니 %p 차이를 같이 보고하세요.')


In [ ]:
# 10-8. Action MSE requires an explicit held-out manifest for each system.
# Never auto-select the last files: those may already be in training data.
from vlapipe.episode import Episode
HOLDOUT_MANIFESTS = {
    'single_vla': Path('data/prepared_single/holdout_single.txt'),
    'hier_vla': Path('data/prepared_hier/holdout_hier.txt'),
}
for name, manifest in HOLDOUT_MANIFESTS.items():
    assert manifest.is_file(), f'{name}: prepare held-out demonstrations before training: {manifest}'

def action_mse(executor, ep):
    adapter = executor.adapter
    adapter.reset()
    states = ep.state_vectors()
    errs = []
    for t in range(len(ep)):
        obs = {k: v[t] for k, v in ep.obs.items()}
        obs['observation.state'] = states[t]
        obs['task'] = ep.task
        action = adapter.select_action(obs)
        errs.append(float(np.mean((action - ep.actions[t]) ** 2)))
    return float(np.mean(errs))

# MSE is a per-model diagnostic, not a direct architecture score:
# whole-task and subtask instruction/data distributions differ.
mse_rows = []
for name, manifest in HOLDOUT_MANIFESTS.items():
    files = [Path(x.strip()) for x in manifest.read_text().splitlines() if x.strip()]
    assert files, f'{manifest}: no held-out episodes'
    for f in files:
        ep = Episode(f)
        mse_rows.append({'system': name, 'episode': f.name,
                         'action_mse': action_mse(SYSTEMS[name][1], ep)})
mse_df = pd.DataFrame(mse_rows)
display(mse_df.groupby('system')['action_mse'].agg(['mean', 'std', 'count']).round(5))


In [ ]:
# 10-9. 그래프 + 제출용 저장
import matplotlib.pyplot as plt
metrics = [('hit_rate', 'Hit-rate'), ('f1', 'F1'), ('wall_s', 'Wall time (s)')]
prompts = list(INSTRUCTIONS)
fig, axes = plt.subplots(len(metrics), len(prompts), figsize=(4.6 * len(prompts), 3.4 * len(metrics)), squeeze=False)
systems_plot = [s for s in ['single_vla', 'hier_vla', 'hier_teacher'] if s in df.system.unique()]
w = 0.8 / len(systems_plot)
for i, (col, label) in enumerate(metrics):
    for j, p in enumerate(prompts):
        ax = axes[i][j]
        for k, s in enumerate(systems_plot):
            m = [df[(df.system == s) & (df.prompt == p) & (df.n_objects == n)][col].mean() for n in N_OBJECTS_LIST]
            e = [df[(df.system == s) & (df.prompt == p) & (df.n_objects == n)][col].std() for n in N_OBJECTS_LIST]
            ax.bar(np.arange(len(N_OBJECTS_LIST)) + k * w, m, w, yerr=e, capsize=2, label=s)
        ax.set_xticks(np.arange(len(N_OBJECTS_LIST)) + w * (len(systems_plot) - 1) / 2)
        ax.set_xticklabels(N_OBJECTS_LIST); ax.set_xlabel('# objects')
        ax.set_title(f'{label} | {p}', fontsize=10)
        if i == 0 and j == 0: ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig('outputs/exp_comparison.png', dpi=150); plt.show()

import shutil, datetime
dst = RESULT_DIR / f'exp_{datetime.datetime.now():%m%d_%H%M}'
dst.mkdir(parents=True, exist_ok=True)
df.to_csv(dst / 'trials.csv', index=False, encoding='utf-8-sig')
agg.to_csv(dst / 'summary.csv', index=False, encoding='utf-8-sig')
cmp_df.to_csv(dst / 'single_vs_hier.csv', index=False, encoding='utf-8-sig')
shutil.copy('outputs/exp_comparison.png', dst / 'exp_comparison.png')
print('저장:', dst)

In [ ]:
# 제출용 저장
import json, shutil, datetime
TEAM_OUT = Path('/content/drive/MyDrive/VLA_Team_2026') if IN_COLAB else PKG.parent
dst = TEAM_OUT / 'results' / f'session13_{datetime.datetime.now():%m%d_%H%M}'
dst.mkdir(parents=True, exist_ok=True)

# 결과 요약 JSON 저장
results_json = {
    'vlm_comparison': summary.to_dict('records') if 'summary' in dir() else [],
    'teacher_results': df_teacher.to_dict('records') if 'df_teacher' in dir() else [],
    'e2e_results': df_e2e.to_dict('records') if 'df_e2e' in dir() else [],
}
with open('outputs/session13_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_json, f, ensure_ascii=False, indent=2, default=str)

for p in Path('outputs').glob('**/*'):
    if p.is_file() and p.suffix in {'.gif', '.png', '.json'}:
        shutil.copy(p, dst / f'{p.parent.name}_{p.name}')
print('저장:', dst)
print(sorted(x.name for x in dst.iterdir()))